In [ ]:
import os
import random
import glob
import librosa
import soundfile as sf
from scipy.signal import correlate
import numpy as np
from math import pi
import torch
import torch.nn as nn
import torchaudio
import itertools
import torch.nn.functional as func
import torchaudio.functional as F
import torchaudio.transforms as T
import matplotlib.pyplot as plt
from scipy.linalg import sqrtm

torch.set_default_dtype(torch.float32)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install frechet_audio_distance
#this library needs to be modified slightly to work with new torch versions....
#https://github.com/gudgud96/frechet-audio-distance
#we use PANN-CNN14 meant for 16k audio
#from frechet_audio_distance import FrechetAudioDistance

ERROR: Could not find a version that satisfies the requirement FrechetAudioDistance (from versions: none)
ERROR: No matching distribution found for FrechetAudioDistance


In [ ]:
def generate_and_save_snippets(file_path, save_dir, num_snippets, segment_duration, sr):
    """Generate fixed-length snippets from a single audio file and save to save_dir."""
    # Load the full audio once
    audio, _ = librosa.load(file_path, sr=sr, mono=True)
    segment_length = int(sr * segment_duration)

    if len(audio) < segment_length:
        raise ValueError(f"Audio file {file_path} is shorter than the segment duration.")

    for i in range(num_snippets):
        start = random.randint(0, len(audio) - segment_length)
        segment = audio[start:start + segment_length]

        snippet_path = save_dir + f"/snippet_{i}.wav"
        sf.write(snippet_path, segment, sr)
    return

In [ ]:
def get_fad(ref_textures,
            syn_textures,
            algo = 'OT',
            gen_refs = False,
            gen_syn = False,
            sr = 16000,
            num_snippets = 128,
            segment_duration = 0.5):
  '''
  '''
  fad_dict = {}
  for i in range(len(ref_textures)):
    ref_texture = ref_textures[i]  # change texture_name
    syn_texture = syn_textures[i]
    texture_name = ref_texture.split('/')[-1].split('.')[0]
    print(f"Processing texture: {texture_name}")
    if not os.path.exists('/content/drive/MyDrive/SoundTextureExperiments/Snippets'):
        os.mkdir('/content/drive/MyDrive/SoundTextureExperiments/Snippets/')
    if not os.path.exists(f'/content/drive/MyDrive/SoundTextureExperiments/Snippets/{texture_name}'):
        os.mkdir(f'/content/drive/MyDrive/SoundTextureExperiments/Snippets/{texture_name}')

    ref_save_dir = f'/content/drive/MyDrive/SoundTextureExperiments/Snippets/{texture_name}/Refs'
    if gen_refs:
      if not os.path.exists(ref_save_dir):
          os.mkdir(ref_save_dir)
      ref_snippets = generate_and_save_snippets(ref_texture, ref_save_dir, num_snippets, segment_duration, sr)

    gen_save_dir = f'/content/drive/MyDrive/SoundTextureExperiments/Snippets/{texture_name}/Results/{algo}'
    if gen_syn:
      if not os.path.exists(f'/content/drive/MyDrive/SoundTextureExperiments/Snippets/{texture_name}/Results'):
          os.mkdir(f'/content/drive/MyDrive/SoundTextureExperiments/Snippets/{texture_name}/Results')
      if not os.path.exists(gen_save_dir):
          os.mkdir(gen_save_dir)
      gen_snippets = generate_and_save_snippets(syn_texture, gen_save_dir, num_snippets, segment_duration, sr)

    fad_calculator = FrechetAudioDistance(model_name="pann",
                                          sample_rate=sr,
                                          preloaded_model_path = '/content/drive/MyDrive/SoundTextureExperiments/',
                                          use_pca=False,
                                          use_activation=False,
                                          verbose=False
    )
    fad_score = fad_calculator.score(
    ref_save_dir,
    gen_save_dir,
    dtype="float32"
)
    print(f"FAD score for texture: {fad_score :.4f}")
    fad_dict[texture_name] = fad_score
  return fad_dict

In [ ]:
def check_texture_similarity(ref_dir, syn_dir, segment_sr=16000, n_bins=64):
    """
    Compute non-copy metrics between reference and synthesized sound texture snippets.

    Args:
        ref_dir (str): Directory containing reference snippets (.wav)
        syn_dir (str): Directory containing synthesized snippets (.wav)
        segment_sr (int): Sampling rate for loading snippets
        n_bins (int): Number of bins for spectral histogram

    Returns:
        dict: {
            'mean_max_corr': mean of max cross-correlation per syn snippet,
            'mean_spec_hist_dist': mean of min spectral histogram distances per syn snippet
        }
    """

    # Load reference snippets
    ref_files = sorted([f for f in os.listdir(ref_dir) if f.endswith('.wav')])
    ref_snippets = [sf.read(os.path.join(ref_dir, f))[0] for f in ref_files]

    # Load synthesized snippets
    syn_files = sorted([f for f in os.listdir(syn_dir) if f.endswith('.wav')])
    syn_snippets = [sf.read(os.path.join(syn_dir, f))[0] for f in syn_files]

    # --- Cross-correlation ---
    max_corrs = []
    for syn in syn_snippets:
        syn = syn - np.mean(syn)
        syn = syn / (np.linalg.norm(syn) + 1e-8)  # normalize
        best_corr = 0
        for ref in ref_snippets:
            ref = ref - np.mean(ref)
            ref = ref / (np.linalg.norm(ref) + 1e-8)
            corr = np.max(correlate(syn, ref, mode='full'))
            best_corr = max(best_corr, corr)
        max_corrs.append(best_corr)

    # --- Spectral histogram distance ---
    spec_dists = []
    for syn in syn_snippets:
        syn_mag = np.abs(librosa.stft(syn))
        syn_hist, _ = np.histogram(syn_mag.flatten(), bins=n_bins, density=True)
        best_dist = np.inf
        for ref in ref_snippets:
            ref_mag = np.abs(librosa.stft(ref))
            ref_hist, _ = np.histogram(ref_mag.flatten(), bins=n_bins, density=True)
            dist = np.linalg.norm(ref_hist - syn_hist)
            best_dist = min(best_dist, dist)
        spec_dists.append(best_dist)

    return {
        'mean_max_corr': float(np.mean(max_corrs)),
        'mean_spec_hist_dist': float(np.mean(spec_dists))
    }

In [ ]:
SR = 16000
NUM_SNIPPETS = 128
SEGMENT_DURATION = 0.5   # seconds

In [ ]:
# ALGO = 'OT'
# ref_textures = glob.glob('/content/drive/MyDrive/SoundTextureExperiments/Refs' + "/*.wav")
# syn_textures = glob.glob(f'/content/drive/MyDrive/SoundTextureExperiments/Results/{ALGO}' + "/*.wav")

# ref_textures.sort()
# syn_textures.sort()

# fad_dict_OT = get_fad(ref_textures,
#             syn_textures,
#             algo = ALGO,
#             gen_refs = False,
#             gen_syn = False,
#             sr = SR,
#             num_snippets = NUM_SNIPPETS,
#             segment_duration = SEGMENT_DURATION)

Processing texture: applause
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
[Frechet Audio Distance] concat_embedding shape: (128, 2048)


/usr/local/lib/python3.12/dist-packages/frechet_audio_distance/fad.py:340: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  covmean, _ = linalg.sqrtm(sigma1.dot(sigma2).astype(complex), disp=False)


FAD score for texture: 1.4689
Processing texture: bees
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
FAD score for texture: 1.1978
Processing texture: birds
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
FAD score for texture: 1.2596
Processing texture: crowd
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
FAD score for texture: 0.8946
Processing texture: fire
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
FAD score for texture: 1.0986
Processing texture: insects
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
FAD score for texture: 1.2332
Processing texture: rain
[Frechet Audio Distance] concat_embedding shape: (128, 

In [ ]:
fad_dict_OT

{'applause': 1.4688686810363505,
 'bees': 1.197807358812657,
 'birds': 1.2596160791263245,
 'crowd': 0.8945901367377882,
 'fire': 1.098563011157168,
 'insects': 1.23319950537822,
 'rain': 1.2203593902942886,
 'sink': 0.6045760905712125,
 'static': 0.6196254000880144,
 'wind': 1.4297547397106496}

In [ ]:
# ALGO = 'AN'
# ref_textures = glob.glob('/content/drive/MyDrive/SoundTextureExperiments/Refs' + "/*.wav")
# syn_textures = glob.glob(f'/content/drive/MyDrive/SoundTextureExperiments/Results/{ALGO}' + "/*.wav")

# ref_textures.sort()
# syn_textures.sort()

# fad_dict_AN = get_fad(ref_textures,
#             syn_textures,
#             algo = ALGO,
#             gen_refs = False,
#             gen_syn = False,
#             sr = SR,
#             num_snippets = NUM_SNIPPETS,
#             segment_duration = SEGMENT_DURATION)

Processing texture: applause
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
[Frechet Audio Distance] concat_embedding shape: (128, 2048)


/usr/local/lib/python3.12/dist-packages/frechet_audio_distance/fad.py:340: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  covmean, _ = linalg.sqrtm(sigma1.dot(sigma2).astype(complex), disp=False)


FAD score for texture: 6.2678
Processing texture: bees
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
FAD score for texture: 2.7855
Processing texture: birds
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
FAD score for texture: 6.3392
Processing texture: crowd
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
FAD score for texture: 3.8127
Processing texture: fire
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
FAD score for texture: 4.4401
Processing texture: insects
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
FAD score for texture: 5.8075
Processing texture: rain
[Frechet Audio Distance] concat_embedding shape: (128, 

In [ ]:
fad_dict_AN

{'applause': 6.267777952424782,
 'bees': 2.7854626006331404,
 'birds': 6.339186616694789,
 'crowd': 3.8126619496660155,
 'fire': 4.4400668472885965,
 'insects': 5.807500001239857,
 'rain': 3.8905421167129295,
 'sink': 11.675113816126512,
 'static': 2.582978447169667,
 'wind': 9.701104178403108}

In [ ]:
# ALGO = 'RISpec'
# ref_textures = glob.glob('/content/drive/MyDrive/SoundTextureExperiments/Refs' + "/*.wav")
# syn_textures = glob.glob(f'/content/drive/MyDrive/SoundTextureExperiments/Results/{ALGO}' + "/*.wav")

# ref_textures.sort()
# syn_textures.sort()

# fad_dict_RISpec = get_fad(ref_textures,
#             syn_textures,
#             algo = ALGO,
#             gen_refs = False,
#             gen_syn = False,
#             sr = SR,
#             num_snippets = NUM_SNIPPETS,
#             segment_duration = SEGMENT_DURATION)

Processing texture: applause
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
[Frechet Audio Distance] concat_embedding shape: (128, 2048)


/usr/local/lib/python3.12/dist-packages/frechet_audio_distance/fad.py:340: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  covmean, _ = linalg.sqrtm(sigma1.dot(sigma2).astype(complex), disp=False)


FAD score for texture: 1.7467
Processing texture: bees
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
FAD score for texture: 2.9981
Processing texture: birds
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
FAD score for texture: 3.7074
Processing texture: crowd
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
FAD score for texture: 2.1788
Processing texture: fire
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
FAD score for texture: 2.0999
Processing texture: insects
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
[Frechet Audio Distance] concat_embedding shape: (128, 2048)
FAD score for texture: 2.0329
Processing texture: rain
[Frechet Audio Distance] concat_embedding shape: (128, 

In [ ]:
fad_dict_RISpec

{'applause': 1.7467025597688082,
 'bees': 2.9980718025527864,
 'birds': 3.70738504811332,
 'crowd': 2.1787592371954716,
 'fire': 2.0998555159985095,
 'insects': 2.0329006008537247,
 'rain': 1.7429353331425386,
 'sink': 2.063604658203216,
 'static': 1.0923035992177588,
 'wind': 4.513845451941097}

# Check Texture Similarity

In [ ]:
ref_textures = glob.glob('/content/drive/MyDrive/SoundTextureExperiments/Refs' + "/*.wav")
ALGO = 'RISpec'
for i in range(len(ref_textures)):
    ref_texture = ref_textures[i].split("/")[-1].split(".")[0]  # change texture_name
    ref_dir = f"/content/drive/MyDrive/SoundTextureExperiments/Snippets/{ref_texture}/Refs"
    syn_dir = f"/content/drive/MyDrive/SoundTextureExperiments/Snippets/applause/Results/{ALGO}"
    metrics = check_texture_similarity(ref_dir, syn_dir)
    print(ref_texture)
    print(metrics)

crowd
{'mean_max_corr': 0.04090777432717646, 'mean_spec_hist_dist': 0.2838990461904085}
sink
{'mean_max_corr': 0.04658343373029537, 'mean_spec_hist_dist': 0.7228331708827038}
static
{'mean_max_corr': 0.08487449259344484, 'mean_spec_hist_dist': 0.4921505753665053}
wind
{'mean_max_corr': 0.037503069545134535, 'mean_spec_hist_dist': 0.2215552061809224}
bees
{'mean_max_corr': 0.06214796625969246, 'mean_spec_hist_dist': 0.3332824851493691}
applause
{'mean_max_corr': 0.7529440462397621, 'mean_spec_hist_dist': 0.026171896285169714}
rain
{'mean_max_corr': 0.07042492180709484, 'mean_spec_hist_dist': 0.5356923468779119}
fire
{'mean_max_corr': 0.025305989562109676, 'mean_spec_hist_dist': 0.167823580323246}
insects
{'mean_max_corr': 0.034796573548310714, 'mean_spec_hist_dist': 0.8932004318044535}
birds
{'mean_max_corr': 0.06217279464674135, 'mean_spec_hist_dist': 0.14113964309067686}


In [ ]:
ref_textures = glob.glob('/content/drive/MyDrive/SoundTextureExperiments/Refs' + "/*.wav")
ALGO = 'OT'
for i in range(len(ref_textures)):
    ref_texture = ref_textures[i].split("/")[-1].split(".")[0]  # change texture_name
    ref_dir = f"/content/drive/MyDrive/SoundTextureExperiments/Snippets/{ref_texture}/Refs"
    syn_dir = f"/content/drive/MyDrive/SoundTextureExperiments/Snippets/applause/Results/{ALGO}"
    metrics = check_texture_similarity(ref_dir, syn_dir)
    print(ref_texture)
    print(metrics)

crowd
{'mean_max_corr': 0.041226308364239475, 'mean_spec_hist_dist': 0.4889465331632912}
sink
{'mean_max_corr': 0.043187404015543544, 'mean_spec_hist_dist': 0.9095319007740876}
static
{'mean_max_corr': 0.08127948132087678, 'mean_spec_hist_dist': 0.28211369780369744}
wind
{'mean_max_corr': 0.041317935489084835, 'mean_spec_hist_dist': 0.3067686843476368}
bees
{'mean_max_corr': 0.0638124472205402, 'mean_spec_hist_dist': 0.5550064450209609}
applause
{'mean_max_corr': 0.7822970423130909, 'mean_spec_hist_dist': 0.05704995215488563}
rain
{'mean_max_corr': 0.06927712334238526, 'mean_spec_hist_dist': 0.6619122825743473}
fire
{'mean_max_corr': 0.023279087486110052, 'mean_spec_hist_dist': 0.2891534151583691}
insects
{'mean_max_corr': 0.03571541011029751, 'mean_spec_hist_dist': 1.095861444870631}
birds
{'mean_max_corr': 0.0605952969169932, 'mean_spec_hist_dist': 0.2665079249083678}


In [ ]:
ref_textures = glob.glob('/content/drive/MyDrive/SoundTextureExperiments/Refs' + "/*.wav")
ALGO = 'AN'
for i in range(len(ref_textures)):
    ref_texture = ref_textures[i].split("/")[-1].split(".")[0]  # change texture_name
    ref_dir = f"/content/drive/MyDrive/SoundTextureExperiments/Snippets/{ref_texture}/Refs"
    syn_dir = f"/content/drive/MyDrive/SoundTextureExperiments/Snippets/applause/Results/{ALGO}"
    metrics = check_texture_similarity(ref_dir, syn_dir)
    print(ref_texture)
    print(metrics)

crowd
{'mean_max_corr': 0.0406974763694527, 'mean_spec_hist_dist': 0.4902754010145822}
sink
{'mean_max_corr': 0.04695371796078991, 'mean_spec_hist_dist': 0.8979393726604428}
static
{'mean_max_corr': 0.08379160579177955, 'mean_spec_hist_dist': 0.43856869657516323}
wind
{'mean_max_corr': 0.035723888430021604, 'mean_spec_hist_dist': 0.3777910607453271}
bees
{'mean_max_corr': 0.0597871525486368, 'mean_spec_hist_dist': 0.5501508127201488}
applause
{'mean_max_corr': 0.1303029102493533, 'mean_spec_hist_dist': 0.16810750384897183}
rain
{'mean_max_corr': 0.07025895224134437, 'mean_spec_hist_dist': 0.5948226687593333}
fire
{'mean_max_corr': 0.022615091798035877, 'mean_spec_hist_dist': 0.38540265301627047}
insects
{'mean_max_corr': 0.03501955739248503, 'mean_spec_hist_dist': 1.0740713588512272}
birds
{'mean_max_corr': 0.06125452893766724, 'mean_spec_hist_dist': 0.2788952494573743}
